In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

ROOT = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

X = pd.read_parquet(DATA_PROCESSED / 'X.parquet')
y = pd.read_parquet(DATA_PROCESSED / 'y.parquet')

2-Rendre le split équilibré

In [ ]:
# Pour chaque cible d'energie, on encode : 0 si nulle, 1 si non-nulle
strate = pd.DataFrame()
strate['has_gas']     = (y['out.natural_gas.total.energy_consumption..kwh']  > 0).astype(int)
strate['has_fueloil'] = (y['out.fuel_oil.total.energy_consumption..kwh']     > 0).astype(int)
strate['has_propane'] = (y['out.propane.total.energy_consumption..kwh']      > 0).astype(int)

# Combinaison en une seule clé lisible : ex. "1_0_0", "0_0_1"
strate_key = (
    strate['has_gas'].astype(str) + '_' +
    strate['has_fueloil'].astype(str) + '_' +
    strate['has_propane'].astype(str)
)

print('Distribution des strates :')
print(strate_key.value_counts())

3_Split

In [ ]:
# Split Train (70%) / Temp (30%) sur l'ensemble complet avec stratification 

X_train, X_temp, y_train, y_temp, strate_train, strate_temp = train_test_split(
    X, y, strate_key,
    test_size=0.30,
    random_state=42,
    stratify=strate_key     
)

# Split Val (15%) / Test (15%) 
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=strate_temp   
)

print(f'Train : {X_train.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')



# Vérifier que les proportions sont bien conservées
def check_split(name, y_split):
    print(f'\n── {name} ──')
    for col in y_split.columns:
        pct_zero = (y_split[col] == 0).mean() * 100
        print(f'  {col[-30:]:<32}  zeros: {pct_zero:5.1f}%')

check_split('TRAIN', y_train)
check_split('VAL',   y_val)
check_split('TEST',  y_test)